In [9]:
from lttb import lttb
import numpy as np
import pandas as pd
import pdb
import plotly.graph_objects as go
import dtat.plot as dtatplot
from time import time
import traceback 

class PlotOrchestrator:
    def __init__(self, data):
        self.data = data
        self.lttb_data = data
        
        # --- FIX 1: The Recursion Guard ---
        # This flag prevents the "Echo" loop.
        self.is_updating = False
        
        self.full_x_start = data["scet"].min()
        self.full_x_end = data["scet"].max()
        
    def lttb(self, n_points, data=None):
        if data is None:
            data = self.data
        
        results = []

        # Group by 'name'
        for name, df in data.groupby("name"):
            df = df.drop_duplicates(subset="scet", keep="first").sort_values("scet")

            x = df["scet"].astype("int64").to_numpy()
            y = df["value"].to_numpy()

            if len(x) <= n_points:
                results.append(df)
                continue
            
            n_samples = min(n_points, len(x))
            data_to_downsample = np.column_stack([x, y])
            
            try:
                x1, y1 = lttb.downsample(data_to_downsample, n_samples).T
            except Exception as e:
                print(f"LTTB failed for {name}: {e}")
                results.append(df)
                continue

            decimated_df = pd.DataFrame({
                "scet": pd.to_datetime(x1),
                "value": y1,
                "name": name,
                "unit": df["unit"].iloc[0]
            })

            results.append(decimated_df)

        if not results:
            return pd.DataFrame()
        return pd.concat(results, ignore_index=True)
    
    def plot(self, data=None):
        if data is None: data = self.data
        fig, c, m, t = dtatplot.make_stacked_graph(
            data, 
            y_vars=[[c] for c in data['name'].unique()], 
            multi_axis=True
            ) 
        return fig
    
    def interactive_lttb_plot(self, n_points=500):
        figw = go.FigureWidget(self.plot(self.lttb(n_points)))
        
        # Listen for changes
        figw.layout.on_change(self.requery_on_zoom, 'xaxis.range', 'xaxis.autorange')
        
        self.fig = figw
        return figw

        
    def requery_on_zoom(self, layout, xrange, autorange):
        # --- FIX 2: Check the Guard ---
        # If we are already updating the plot, ignore this event.
        # This stops the infinite loop where updating the plot triggers a new update.
        if self.is_updating:
            return

        self.is_updating = True

        try:
            # 1. Determine Window
            if (autorange is True) or (xrange is None):
                # RESET
                x_start = self.full_x_start
                x_end = self.full_x_end
                
                # --- FIX 3: Unlatch Autorange ---
                # We manually reset autorange to False so the NEXT zoom isn't ignored.
                # We do this in the layout update below.
                force_autorange_off = True
            else:
                # ZOOM
                x_start = pd.to_datetime(xrange[0])
                x_end = pd.to_datetime(xrange[1])
                force_autorange_off = False
                
                # Timezone safety
                data_tz = getattr(self.data["scet"].dtype, 'tz', None)
                if data_tz is None and getattr(x_start, 'tz', None) is not None:
                    x_start = x_start.tz_localize(None)
                    x_end = x_end.tz_localize(None)

            # 2. Filter & Resample
            zoomed_data = self.data[(self.data["scet"] >= x_start) & (self.data["scet"] <= x_end)]
            
            if len(zoomed_data) == 0:
                return

            decimated_zoomed = self.lttb(500, zoomed_data)

            # 3. Update Plot
            with self.fig.batch_update():
                # If we just handled a "Home" click, strictly turn off autorange 
                # so the widget knows we are now in control of the range again.
                if force_autorange_off:
                    self.fig.layout.xaxis.autorange = False

                for trace in self.fig.data:
                    trace_name = trace.name
                    df_trace = decimated_zoomed[decimated_zoomed['name'] == trace_name]
                    
                    if df_trace.empty: continue

                    trace.x = df_trace['scet']
                    trace.y = df_trace['value']
                    
        except Exception:
            traceback.print_exc()
        finally:
            # --- FIX 4: Release the Guard ---
            # Always ensure we allow future updates, even if code crashed above.
            self.is_updating = False

In [10]:
n_points = 100000
scet = np.linspace(0, 200 * np.pi, n_points)
value = np.sin(scet)

data = pd.DataFrame({
    'scet':  [pd.to_datetime(x*1e11) for x in scet] + [pd.to_datetime(x*1e11) for x in scet],
    'value': [np.sin(x) for x in scet] + [np.cos(x) for x in scet],
    'name': ['sin']*n_points + ['cos']*n_points,
    'unit': ['NA']*2*n_points
})

po = PlotOrchestrator(data)

po.interactive_lttb_plot(500)

FigureWidget({
    'data': [{'hovertemplate': ('X: scet: %{x|%Y-%jT%H:%M:%S.%L' ... '%jT%H:%M:%S.%L}<extra></extra>'),
              'marker': {'color': '#0078AF',
                         'colorscale': [[0.0, 'rgb(0,0,131)'], [0.2,
                                        'rgb(0,60,170)'], [0.4, 'rgb(5,255,255)'],
                                        [0.6, 'rgb(255,255,0)'], [0.8,
                                        'rgb(250,0,0)'], [1.0, 'rgb(128,0,0)']],
                         'line': {'color': '#0073FF', 'width': 0.5},
                         'showscale': False,
                         'size': 5,
                         'symbol': 'circle'},
              'meta': [[None, Timestamp('1970-01-01 00:00:00')], [None,
                       Timestamp('1970-01-01 00:01:15.398977675')], [None,
                       Timestamp('1970-01-01 00:03:58.135104493')], ..., [None,
                       Timestamp('1970-01-01 17:23:23.142839512')], [None,
                       Timestamp('

In [3]:
po.lttb(500, po.data)

,scet,value,name,unit
0,1970-01-01 00:00:00.000000,1.000000e+00,cos,NA
1,1970-01-01 00:01:15.398977,7.289635e-01,cos,NA
2,1970-01-01 00:03:58.135104,-7.246695e-01,cos,NA
3,1970-01-01 00:05:36.782100,-9.745193e-01,cos,NA
4,1970-01-01 00:06:39.614581,-6.565556e-01,cos,NA
...,...,...,...,...
995,1970-01-01 17:19:57.680625,9.320868e-01,sin,NA
996,1970-01-01 17:22:59.894821,-5.826700e-01,sin,NA
997,1970-01-01 17:24:36.656842,-9.998226e-01,sin,NA
998,1970-01-01 17:25:44.515922,-7.664986e-01,sin,NA


In [4]:
po.xmin

AttributeError: 'PlotOrchestrator' object has no attribute 'xmin'

In [ ]:
po.last_x_end

In [ ]:
po.last_x_start

In [ ]:
po.last_x_end